In [3]:
import sys
from pathlib import Path

# Get the absolute path of the parent directory of your current working directory
ROOT = Path.cwd().resolve().parent

# Add this parent directory to the top of Python's module search path
sys.path.insert(0, str(ROOT))


In [ ]:
import shutil
import re
import pandas as pd
from pathlib import Path
from src.utils.helpers import totalizador, planilha_lancamento, normalizar_parc, corrigir_xls_html_para_xlsx

arquivo = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT\0626_FECFIN_CEMAF PART.xlsx")
origem_lanc = Path(r"C:\Users\manja\OneDrive\Documentos\AutoExtrato\data\Lancamentos_Contabeis.xlsm")
destino_lanc = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT")

with pd.ExcelFile(arquivo) as xls:
    abas = xls.sheet_names

    if "CEMAF PART" in arquivo.stem:
        bancos = ["UNICRED", "CAIXA"]
        extratos = [aba for aba in abas for banco in bancos if banco in aba]

        for extrato in extratos:
            df = pd.read_excel(arquivo, sheet_name=extrato)
            
            if "UNICRED" in extrato:
                df.columns = df.iloc[5] # Definindo cabeçalho das colunas
                df = df[6:].reset_index(drop=True) # Removendo as primeiras 6 linhas e resetando o índice
            elif "CAIXA" in extrato:
                df.columns = df.iloc[4] # Definindo cabeçalho das colunas
                df = df[5:].reset_index(drop=True) # Removendo linhas acima do cabeçalho e resetando o índice       

            


print(abas)
print(extratos)
display(df)


['RESUMO', 'NFS FORNEC', 'FAT X REC', 'UNICRED 11297-6', 'CAIXA']
['UNICRED 11297-6', 'CAIXA']


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,PROTOCOLO CONTABILIDADE,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CAIXA:,JUNHO 2026,NaN,CEMAF PARTICIPAÇÕES,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DATA,TIPO,Nº DOC,HISTÓRICO,ENTRADA,SAIDA,SALDO,OBS
5,2026-06-01 00:00:00,NaN,NaN,SALDO ANTERIOR,0,0,0,NaN
6,2026-06-30 00:00:00,NaN,NaN,SALDO FINAL,0,0,0,NaN


In [34]:
df.columns = df.iloc[4] # Definindo cabeçalho das colunas
df = df[5:].reset_index(drop=True) # Removendo linhas acima do cabeçalho e resetando o índice
df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['OBS']} {x['TIPO']} {x['Nº DOC']} {x['HISTÓRICO']}", axis=1)
df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.strip().str.upper()
df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAIDA"]]
df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
df["SAIDA"] = pd.to_numeric(df["SAIDA"], errors="coerce").fillna(0)
df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAIDA"] * -1)
df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]
df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
df["VALOR"] = df["VALOR"].abs()
df["DATA"] = df["DATA"].apply(lambda x: pd.to_datetime(x, errors="coerce").strftime("%d/%m/%Y") if pd.notnull(x) else "")

display(df)

4,DATA,DESCRIÇÃO,VALOR,TIPO


In [28]:
df.columns = df.iloc[5] # Definindo cabeçalho das colunas
df = df[6:].reset_index(drop=True) # Removendo as primeiras 6 linhas e resetando o índice
df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['OBS']} {x['TIPO']} {x['Nº DOC']} {x['HISTÓRICO']}", axis=1)
df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.strip().str.upper()
df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAIDA"]]
df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
df["SAIDA"] = pd.to_numeric(df["SAIDA"], errors="coerce").fillna(0)
df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAIDA"] * -1)
df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]
df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
df["VALOR"] = df["VALOR"].abs()
df["DATA"] = df["DATA"].apply(lambda x: pd.to_datetime(x, errors="coerce").strftime("%d/%m/%Y") if pd.notnull(x) else "")

display(df)

5,DATA,DESCRIÇÃO,VALOR,TIPO
1,16/06/2026,APORT SOC ADP PARTICIPAÇÕES,3900,C
2,16/06/2026,APORT SOC ALOHA PARTT,10800,C
3,16/06/2026,INTEGRALIZAÇÃO CAPITAL UNICRED,100,D
4,17/06/2026,PAGAMENTO AVIAO HENRIQUE CELSO,14500,D
5,17/06/2026,APORT SOC CE PARTICIPAÇÕES,10000,C
6,17/06/2026,PAGAMENTO AVIAO HENRIQUE CELSO,10000,D
7,17/06/2026,APORT SOC CE PARTICIPAÇÕES,5300,C
8,17/06/2026,APORT SOC CARLOS EDUARDO DAYRELL,100,C
9,17/06/2026,PAGAMENTO AVIAO HENRIQUE CELSO,5500,D


# FECFIN

In [ ]:
import shutil
import re
import pandas as pd
from pathlib import Path
from src.utils.helpers import totalizador, planilha_lancamento, normalizar_parc

arquivo = Path(r"G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\501_EBLUX\Extratos\26\05\0526_FECFIN CAIXA_EBLUX.xlsx")

origem_lanc = Path(r"C:\Users\manja\OneDrive\Documentos\AutoExtrato\data\Lancamentos_Contabeis.xlsm")
destino_lanc = Path(r"G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\501_EBLUX\Extratos\26\05")

layout = ['Data Competência\n', 'Cliente / Fornecedor / Funcionario ', 'Conta',
        'Descrição', 'Banco', 'Forma de Pagamento', 'Parc', 'Valor',
        'Data Pagamento | Recebimento\n', 'NF', 'Pedido', 'CONC',
        'Entrada | Saida']

df = pd.read_excel(arquivo, header=4, converters={"Parc": normalizar_parc})
df["DESCRIÇÃO"] = df.apply(lambda x: " ".join(filter(None, [f"NF {str(int(x['NF'])) if pd.notna(x['NF']) and isinstance(x['NF'], float) and x['NF'].is_integer() else str(x['NF']).replace('.0', '').strip()}" if pd.notna(x["NF"]) and str(x["NF"]).strip() != "" else "", str(x["Descrição"]), str(x["Cliente / Fornecedor / Funcionario "]), str(x["Parc"]), str(x["Conta"])])), axis=1)
df["TIPO"] = df.apply(lambda x: "C" if x["Entrada | Saida"] == "Entrada" else "D", axis=1)
df = df.rename(columns={"Data Pagamento | Recebimento\n": "DATA", "Valor": "VALOR"})
df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO", "Banco"]]
df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.strip().str.upper()
df["DATA"] = df["DATA"].dt.strftime("%d/%m/%Y")
df["VALOR"] = df["VALOR"].abs()
banco = str(df["Banco"].iloc[0]).upper()
df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]

if banco in arquivo.stem:
    arquivo_lanc = destino_lanc / f"[LANC] {arquivo.stem}.xlsm"
else:
    arquivo_lanc = destino_lanc / f"[LANC] {arquivo.stem}_{banco}.xlsm"

shutil.copy2(origem_lanc, arquivo_lanc)
planilha_lancamento(df, arquivo_lanc)
 
display(df)


# FECFIN CEMAF PART (UNICRED / CAIXA)

In [ ]:
import shutil
import re
import pandas as pd
from pathlib import Path
from src.utils.helpers import totalizador, planilha_lancamento, normalizar_parc, corrigir_xls_html_para_xlsx

arquivo = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT\0626_FECFIN_CEMAF PART.xlsx")
origem_lanc = Path(r"C:\Users\manja\OneDrive\Documentos\AutoExtrato\data\Lancamentos_Contabeis.xlsm")
destino_lanc = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT")

with pd.ExcelFile(arquivo) as xls:
    abas = xls.sheet_names

    if "CEMAF PART" in arquivo.stem:
        bancos = ["UNICRED", "CAIXA"]
        extratos = [aba for aba in abas for banco in bancos if banco in aba]

        for extrato in extratos:
            df = pd.read_excel(arquivo, sheet_name=extrato)

            if "UNICRED" in extrato:
                df.columns = df.iloc[5] # Definindo cabeçalho das colunas
                df = df[6:].reset_index(drop=True) # Removendo as primeiras 6 linhas e resetando o índice
            elif "CAIXA" in extrato:
                df.columns = df.iloc[4] # Definindo cabeçalho das colunas
                df = df[5:].reset_index(drop=True) # Removendo linhas acima do cabeçalho e resetando o índice       

            if df.empty:
                print(f"A planilha '{extrato}' está vazia. Pulando para a próxima.")
                continue

            df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['OBS']} {x['TIPO']} {x['Nº DOC']} {x['HISTÓRICO']}", axis=1)
            df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.strip().str.upper()
            df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAIDA"]]
            df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
            df["SAIDA"] = pd.to_numeric(df["SAIDA"], errors="coerce").fillna(0)
            df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAIDA"] * -1)
            df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]
            df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
            df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
            df["VALOR"] = df["VALOR"].abs()
            df["DATA"] = df["DATA"].apply(lambda x: pd.to_datetime(x, errors="coerce").strftime("%d/%m/%Y") if pd.notnull(x) else "")

            if not df.empty:
                arquivo_lanc = destino_lanc / f"[LANC] {arquivo.stem}_{extrato}.xlsm"
                shutil.copy2(origem_lanc, arquivo_lanc)
                planilha_lancamento(df, arquivo_lanc)
